# LeetCode 1584 — Min Cost to Connect All Points bằng Python

Link đề: https://leetcode.com/problems/min-cost-to-connect-all-points/ · Độ khó: **Medium** · Chủ đề: Array, Union-Find, Graph Theory, Minimum Spanning Tree, Prim's Algorithm, Kruskal's Algorithm, Borůvka's Algorithm

Skeleton LeetCode:

```python
class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra bài toán là **cây khung nhỏ nhất (Minimum Spanning Tree — MST)** trên **đồ thị đầy đủ** với trọng số là **khoảng cách Manhattan**.
2. Viết baseline **vét cạn mọi tập `n-1` cạnh** để hiểu định nghĩa cây khung và có chuẩn đối chiếu.
3. Cài đặt **Prim `O(n²)` bằng mảng** — lựa chọn tốt nhất cho đồ thị dày.
4. Cài đặt **Kruskal + Union-Find** (nén đường đi, hợp theo kích thước) và **Prim dùng heap**, so sánh ưu nhược điểm.
5. Hiểu **tính chất lát cắt (cut property)** — lý do mọi thuật toán tham lam MST đều đúng.

## 2. Đề bài & ví dụ

Cho mảng `points` biểu diễn toạ độ nguyên của các điểm trên mặt phẳng 2D, `points[i] = [xᵢ, yᵢ]`.

Chi phí nối hai điểm `[xᵢ, yᵢ]` và `[xⱼ, yⱼ]` là **khoảng cách Manhattan**: `|xᵢ − xⱼ| + |yᵢ − yⱼ|`.

Trả về **chi phí nhỏ nhất** để nối tất cả các điểm. Tất cả các điểm được nối nếu giữa hai điểm bất kỳ có **đúng một đường đi đơn**.

**Ví dụ 1:** `points = [[0,0],[2,2],[3,10],[5,2],[7,0]]` → `20`.

**Ví dụ 2:** `points = [[3,12],[-2,5],[-4,1]]` → `18`.

**Ràng buộc:**
- `1 ≤ points.length ≤ 1000`
- `−10⁶ ≤ xᵢ, yᵢ ≤ 10⁶`
- Mọi cặp `(xᵢ, yᵢ)` phân biệt.

"Đúng một đường đi đơn giữa mọi cặp điểm" ⇔ tập cạnh được chọn tạo thành một **cây** phủ mọi điểm (cây khung).

In [1]:
import heapq
import random
import time
from itertools import combinations
from typing import List

examples = [([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]], 20), ([[3, 12], [-2, 5], [-4, 1]], 18)]
for pts, want in examples:
    print(f"points = {pts} -> mong đợi {want}")

points = [[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]] -> mong đợi 20
points = [[3, 12], [-2, 5], [-4, 1]] -> mong đợi 18


## 3. Kiến thức nền

### 3.1 Khoảng cách Manhattan

`dist(p, q) = |p.x − q.x| + |p.y − q.y|` — quãng đường đi theo lưới ô vuông (chỉ đi ngang/dọc).

In [2]:
def manhattan(p, q):
    return abs(p[0] - q[0]) + abs(p[1] - q[1])

pts = [[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]
for i, j in [(0, 1), (1, 3), (3, 4), (1, 2)]:
    print(f"dist({pts[i]}, {pts[j]}) = {manhattan(pts[i], pts[j])}")

dist([0, 0], [2, 2]) = 4
dist([2, 2], [5, 2]) = 3
dist([5, 2], [7, 0]) = 4
dist([2, 2], [3, 10]) = 9


### 3.2 Cây khung (spanning tree) và cây khung nhỏ nhất

Với đồ thị liên thông `n` đỉnh:
- **Cây khung** = tập cạnh **liên thông mọi đỉnh** và **không có chu trình** ⇔ liên thông và có đúng `n − 1` cạnh.
- **MST** = cây khung có **tổng trọng số nhỏ nhất**.

Ở bài này, giữa **mọi cặp điểm** đều có thể nối ⇒ đồ thị đầy đủ với `n(n−1)/2` cạnh (tối đa ~`5·10⁵` cạnh khi `n = 1000`).

In [3]:
n = 5
print("số cạnh của đồ thị đầy đủ n=5:", n * (n - 1) // 2, "| n=1000:", 1000 * 999 // 2)
print("một cây khung luôn có n-1 =", n - 1, "cạnh")

số cạnh của đồ thị đầy đủ n=5: 10 | n=1000: 499500
một cây khung luôn có n-1 = 4 cạnh


### 3.3 Union-Find (Disjoint Set Union)

Quản lý các **thành phần liên thông** khi thêm cạnh:
- `find(x)`: tìm gốc đại diện của tập chứa `x` (kèm **nén đường đi**).
- `union(a, b)`: gộp hai tập; trả `False` nếu `a`, `b` đã cùng tập (thêm cạnh sẽ tạo chu trình).
- **Hợp theo kích thước** giữ cây nông ⇒ gần như `O(1)` mỗi thao tác.

In [4]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # nén đường đi (halving)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True

d = DSU(5)
print(d.union(0, 1), d.union(3, 4), d.union(1, 3), d.union(0, 4))   # cạnh cuối tạo chu trình -> False
print("gốc của từng đỉnh:", [d.find(i) for i in range(5)])

True True True False
gốc của từng đỉnh: [0, 0, 2, 0, 0]


### 3.4 `heapq` — hàng đợi ưu tiên (min-heap)

`heapq.heappush(h, (khoá, giá trị))` và `heapq.heappop(h)` lấy phần tử có khoá nhỏ nhất trong `O(log k)`. Dùng cho Prim bản heap.

In [5]:
h = []
for item in [(7, "a"), (2, "b"), (5, "c")]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(3)])

[(2, 'b'), (5, 'c'), (7, 'a')]


### 3.5 `itertools.combinations` để vét cạn tập cạnh

`combinations(edges, k)` liệt kê mọi tập con `k` phần tử — dùng cho brute force với `n` rất nhỏ.

In [6]:
edges = [(0, 1), (0, 2), (1, 2)]
print(list(combinations(edges, 2)))

[((0, 1), (0, 2)), ((0, 1), (1, 2)), ((0, 2), (1, 2))]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `points = [[3,12],[-2,5],[-4,1]]`. Ba cạnh:
- `0–1`: `|3−(−2)| + |12−5| = 5 + 7 = 12`
- `0–2`: `7 + 11 = 18`
- `1–2`: `2 + 4 = 6`

Cây khung có 2 cạnh; ba lựa chọn: `{12, 18} = 30`, `{12, 6} = 18`, `{18, 6} = 24` ⇒ nhỏ nhất **18**. ✔

Trực giác tham lam: **cạnh rẻ nhất `1–2` (6) chắc chắn nên lấy**; sau đó cần nối điểm 0 vào nhóm `{1, 2}` bằng cạnh rẻ nhất băng qua: `min(12, 18) = 12`. Đây chính là ý tưởng của cả Kruskal (lấy cạnh rẻ nhất không tạo chu trình) lẫn Prim (mở rộng cây bằng cạnh rẻ nhất ra ngoài).

In [7]:
pts = [[3, 12], [-2, 5], [-4, 1]]
all_edges = [(manhattan(pts[i], pts[j]), i, j) for i, j in combinations(range(3), 2)]
print("các cạnh (w, i, j):", all_edges)
for pair in combinations(all_edges, 2):
    print("chọn", pair, "-> tổng", sum(w for w, _, _ in pair))

các cạnh (w, i, j): [(12, 0, 1), (18, 0, 2), (6, 1, 2)]
chọn ((12, 0, 1), (18, 0, 2)) -> tổng 30
chọn ((12, 0, 1), (6, 1, 2)) -> tổng 18
chọn ((18, 0, 2), (6, 1, 2)) -> tổng 24


## 5. Cách 1 — Brute Force / baseline: thử mọi tập `n − 1` cạnh

**Ý tưởng:** liệt kê mọi tập `n − 1` cạnh trong `n(n−1)/2` cạnh; tập nào **không tạo chu trình** (kiểm bằng Union-Find) thì là cây khung; lấy tổng nhỏ nhất.

In [8]:
def brute_force(points: List[List[int]]) -> int:
    n = len(points)
    if n == 1:
        return 0
    edges = [(manhattan(points[i], points[j]), i, j) for i, j in combinations(range(n), 2)]
    best = None
    for subset in combinations(edges, n - 1):
        d = DSU(n)
        if all(d.union(i, j) for _, i, j in subset):     # n-1 cạnh không chu trình => cây khung
            cost = sum(w for w, _, _ in subset)
            best = cost if best is None else min(best, cost)
    return best

for pts, want in examples:
    print(brute_force(pts), "| mong đợi", want)

20 | mong đợi 20
18 | mong đợi 18


**Độ phức tạp:** `C(n(n−1)/2, n−1)` tập con — bùng nổ cực nhanh (`n = 7`: `C(21, 6) = 54 264`; `n = 10`: `C(45, 9) ≈ 8.9·10⁹`). Chỉ dùng cho `n ≤ 7`.

**Nút thắt:** thử mọi tổ hợp là thừa — tính chất lát cắt (phần 6) cho phép **chọn tham lam từng cạnh** mà không cần quay lui.

In [9]:
rng = random.Random(1)
pts7 = [[rng.randint(-20, 20), rng.randint(-20, 20)] for _ in range(7)]
pts7 = [list(p) for p in {tuple(p) for p in pts7}]
t0 = time.perf_counter()
r = brute_force(pts7)
print(f"n={len(pts7)}: {r}, mất {time.perf_counter() - t0:.3f}s (n=10 sẽ cần hàng tỷ tổ hợp)")

n=7: 73, mất 0.119s (n=10 sẽ cần hàng tỷ tổ hợp)


## 6. Ý tưởng thuật toán / bước đột phá

**Tính chất lát cắt (cut property):** chia tập đỉnh thành hai phần `S` và `V \ S` (không rỗng). Cạnh **nhẹ nhất** băng qua lát cắt (một đầu trong `S`, một đầu ngoài) luôn thuộc **một** MST nào đó.

Từ đó có hai chiến lược tham lam:

- **Prim:** bắt đầu từ một đỉnh; `S` = các đỉnh đã vào cây. Mỗi bước thêm cạnh nhẹ nhất từ `S` ra ngoài. Duy trì `dist[v]` = cạnh rẻ nhất nối `v` (chưa vào cây) với cây.
  - *Nếu* đồ thị **dày** (`m ≈ n²`) *thì* dùng mảng `dist` và quét tuyến tính tìm min: `O(n²)` — **tối ưu cho bài này**.
  - *Nếu* đồ thị thưa *thì* dùng heap: `O(m log n)`.
- **Kruskal:** sắp mọi cạnh tăng dần; lần lượt thêm cạnh nếu nó **không tạo chu trình** (Union-Find). Dừng khi đủ `n − 1` cạnh. `O(m log m)` vì sắp xếp; với `m = n²/2` là `O(n² log n)`.

Lưu ý: bài **không** cho danh sách cạnh — ta tự sinh trọng số "theo yêu cầu" từ toạ độ. Prim `O(n²)` thậm chí **không cần lưu** cạnh nào (bộ nhớ `O(n)`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prim `O(n²)` với mảng `dist` (tối ưu cho đồ thị dày)

**Đoạn 1:** `dist[v] = ∞` cho mọi `v`, `dist[0] = 0`; `in_tree[v] = False`.

**Đoạn 2:** lặp `n` lần:
- chọn `u` **chưa vào cây** có `dist[u]` nhỏ nhất (quét `O(n)`);
- đưa `u` vào cây, cộng `dist[u]` vào đáp án;
- với mọi `v` chưa vào cây, cập nhật `dist[v] = min(dist[v], manhattan(u, v))` (`O(n)`).

In [10]:
def mst_prim_array(points: List[List[int]]) -> int:
    n = len(points)
    INF = float("inf")
    dist = [INF] * n
    dist[0] = 0
    in_tree = [False] * n
    total = 0
    for _ in range(n):
        u, best = -1, INF
        for v in range(n):
            if not in_tree[v] and dist[v] < best:
                u, best = v, dist[v]
        in_tree[u] = True
        total += best
        ux, uy = points[u]
        for v in range(n):
            if not in_tree[v]:
                d = abs(ux - points[v][0]) + abs(uy - points[v][1])
                if d < dist[v]:
                    dist[v] = d
    return total

for pts, want in examples:
    print(mst_prim_array(pts), "| mong đợi", want)

20 | mong đợi 20
18 | mong đợi 18


Code nộp LeetCode cho cách 2 (giống phần 9):

```python
class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        INF = float("inf")
        dist = [INF] * n
        dist[0] = 0
        in_tree = [False] * n
        total = 0
        for _ in range(n):
            u, best = -1, INF
            for v in range(n):
                if not in_tree[v] and dist[v] < best:
                    u, best = v, dist[v]
            in_tree[u] = True
            total += best
            ux, uy = points[u]
            for v in range(n):
                if not in_tree[v]:
                    d = abs(ux - points[v][0]) + abs(uy - points[v][1])
                    if d < dist[v]:
                        dist[v] = d
        return total
```

**Độ phức tạp:** `O(n²)` thời gian, `O(n)` bộ nhớ (không lưu cạnh).

### 7.2 Cách 3 — Kruskal + Union-Find

**Đoạn 1:** sinh mọi cạnh `(w, i, j)` với `i < j` — `n(n−1)/2` cạnh.

**Đoạn 2:** sắp tăng dần theo `w`.

**Đoạn 3:** duyệt cạnh; nếu `union(i, j)` thành công (khác thành phần) thì cộng `w`, đếm cạnh; dừng khi đủ `n − 1`.

In [11]:
def mst_kruskal(points: List[List[int]]) -> int:
    n = len(points)
    edges = []
    for i in range(n):
        xi, yi = points[i]
        for j in range(i + 1, n):
            edges.append((abs(xi - points[j][0]) + abs(yi - points[j][1]), i, j))
    edges.sort()
    d = DSU(n)
    total = used = 0
    for w, i, j in edges:
        if used == n - 1:
            break
        if d.union(i, j):
            total += w
            used += 1
    return total

for pts, want in examples:
    print(mst_kruskal(pts), "| mong đợi", want)

20 | mong đợi 20
18 | mong đợi 18


Code nộp LeetCode cho cách 3 (Union-Find viết gọn bên trong):

```python
class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        edges = sorted(
            (abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1]), i, j)
            for i in range(n) for j in range(i + 1, n)
        )
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        total = used = 0
        for w, i, j in edges:
            ri, rj = find(i), find(j)
            if ri != rj:
                parent[ri] = rj
                total += w
                used += 1
                if used == n - 1:
                    break
        return total
```

**Độ phức tạp:** `O(n² log n)` thời gian (sắp `~n²/2` cạnh), `O(n²)` bộ nhớ cho danh sách cạnh. Chậm và tốn bộ nhớ hơn Prim mảng ở bài này, nhưng là thuật toán quen thuộc nhất và phù hợp khi **đề cho sẵn danh sách cạnh thưa**.

### 7.3 Cách 4 — Prim dùng heap (lazy)

Giống Prim nhưng thay việc quét tìm min bằng heap chứa `(khoảng cách, đỉnh)`. "Lazy": không xoá phần tử cũ trong heap; khi pop ra đỉnh **đã vào cây** thì bỏ qua.

Với đồ thị đầy đủ, mỗi lần thêm đỉnh ta đẩy tới `n` phần tử ⇒ heap có thể tới `O(n²)` phần tử ⇒ `O(n² log n)`. Có thể giảm bằng cách chỉ đẩy khi `d < dist[v]`.

In [12]:
def mst_prim_heap(points: List[List[int]]) -> int:
    n = len(points)
    INF = float("inf")
    dist = [INF] * n
    dist[0] = 0
    in_tree = [False] * n
    heap = [(0, 0)]
    total = added = 0
    while heap and added < n:
        d, u = heapq.heappop(heap)
        if in_tree[u]:
            continue                    # bản ghi cũ, bỏ qua
        in_tree[u] = True
        total += d
        added += 1
        ux, uy = points[u]
        for v in range(n):
            if not in_tree[v]:
                nd = abs(ux - points[v][0]) + abs(uy - points[v][1])
                if nd < dist[v]:        # chỉ đẩy khi tốt hơn -> heap nhỏ hơn
                    dist[v] = nd
                    heapq.heappush(heap, (nd, v))
    return total

for pts, want in examples:
    print(mst_prim_heap(pts), "| mong đợi", want)

20 | mong đợi 20
18 | mong đợi 18


Code nộp LeetCode cho cách 4:

```python
class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        dist = [float("inf")] * n
        dist[0] = 0
        in_tree = [False] * n
        heap = [(0, 0)]
        total = added = 0
        while heap and added < n:
            d, u = heapq.heappop(heap)
            if in_tree[u]:
                continue
            in_tree[u] = True
            total += d
            added += 1
            ux, uy = points[u]
            for v in range(n):
                if not in_tree[v]:
                    nd = abs(ux - points[v][0]) + abs(uy - points[v][1])
                    if nd < dist[v]:
                        dist[v] = nd
                        heapq.heappush(heap, (nd, v))
        return total
```

**Độ phức tạp:** `O(n² log n)` xấu nhất, `O(n²)` bộ nhớ xấu nhất cho heap. Ở đồ thị đầy đủ, bản mảng (cách 2) tốt hơn; bản heap tỏa sáng khi đồ thị **thưa** và có danh sách kề.

In [13]:
# So sánh thời gian ba cách trên n = 1000 điểm ngẫu nhiên (giới hạn của đề)
rng = random.Random(1000)
pts_big = set()
while len(pts_big) < 1000:
    pts_big.add((rng.randint(-10**6, 10**6), rng.randint(-10**6, 10**6)))
pts_big = [list(p) for p in pts_big]
results = {}
for f in [mst_prim_array, mst_kruskal, mst_prim_heap]:
    t0 = time.perf_counter()
    results[f.__name__] = f(pts_big)
    print(f"{f.__name__:<15} {results[f.__name__]:>12}  {time.perf_counter() - t0:.3f}s")
assert len(set(results.values())) == 1

mst_prim_array      52461280  0.128s


mst_kruskal         52461280  0.423s
mst_prim_heap       52461280  0.100s


## 8. TRACE từng bước

Prim mảng với ví dụ 1: `P0=(0,0), P1=(2,2), P2=(3,10), P3=(5,2), P4=(7,0)`.

| bước | chọn u (dist) | dist sau cập nhật `[P0, P1, P2, P3, P4]` | total |
|---|---|---|---|
| 1 | P0 (0) | `[✓, 4, 13, 7, 7]` | 0 |
| 2 | P1 (4) | `[✓, ✓, 9, 3, 7]` | 4 |
| 3 | P3 (3) | `[✓, ✓, 9, ✓, 4]` | 7 |
| 4 | P4 (4) | `[✓, ✓, 9, ✓, ✓]` | 11 |
| 5 | P2 (9) | xong | **20** |

Cây: `P0–P1 (4)`, `P1–P3 (3)`, `P3–P4 (4)`, `P1–P2 (9)`.

In [14]:
def trace_prim(points):
    n = len(points)
    INF = float("inf")
    dist, parent = [INF] * n, [-1] * n
    dist[0] = 0
    in_tree = [False] * n
    total = 0
    for step in range(1, n + 1):
        u = min((v for v in range(n) if not in_tree[v]), key=lambda v: dist[v])
        in_tree[u] = True
        total += dist[u]
        for v in range(n):
            if not in_tree[v]:
                d = manhattan(points[u], points[v])
                if d < dist[v]:
                    dist[v], parent[v] = d, u
        shown = ["✓" if in_tree[v] else str(dist[v]) for v in range(n)]
        edge = f"cạnh P{parent[u]}-P{u}" if parent[u] != -1 else "gốc"
        print(f"bước {step}: thêm P{u} ({edge}, w={dist[u] if parent[u] != -1 else 0}) | dist={shown} | total={total}")
    return total

trace_prim([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]])

bước 1: thêm P0 (gốc, w=0) | dist=['✓', '4', '13', '7', '7'] | total=0
bước 2: thêm P1 (cạnh P0-P1, w=4) | dist=['✓', '✓', '9', '3', '7'] | total=4
bước 3: thêm P3 (cạnh P1-P3, w=3) | dist=['✓', '✓', '9', '✓', '4'] | total=7
bước 4: thêm P4 (cạnh P3-P4, w=4) | dist=['✓', '✓', '9', '✓', '✓'] | total=11
bước 5: thêm P2 (cạnh P1-P2, w=9) | dist=['✓', '✓', '✓', '✓', '✓'] | total=20


20

In [15]:
def trace_kruskal(points):
    n = len(points)
    edges = sorted((manhattan(points[i], points[j]), i, j) for i, j in combinations(range(n), 2))
    d = DSU(n)
    total = used = 0
    for w, i, j in edges:
        if used == n - 1:
            break
        ok = d.union(i, j)
        if ok:
            total += w
            used += 1
        print(f"cạnh P{i}-P{j} w={w:>2}: {'NHẬN' if ok else 'bỏ (tạo chu trình)'} | total={total} | số cạnh={used}")
    return total

trace_kruskal([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]])

cạnh P1-P3 w= 3: NHẬN | total=3 | số cạnh=1
cạnh P0-P1 w= 4: NHẬN | total=7 | số cạnh=2
cạnh P3-P4 w= 4: NHẬN | total=11 | số cạnh=3
cạnh P0-P3 w= 7: bỏ (tạo chu trình) | total=11 | số cạnh=3
cạnh P0-P4 w= 7: bỏ (tạo chu trình) | total=11 | số cạnh=3
cạnh P1-P4 w= 7: bỏ (tạo chu trình) | total=11 | số cạnh=3
cạnh P1-P2 w= 9: NHẬN | total=20 | số cạnh=4


20

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        INF = float("inf")
        dist = [INF] * n
        dist[0] = 0
        in_tree = [False] * n
        total = 0
        for _ in range(n):
            u, best = -1, INF
            for v in range(n):
                if not in_tree[v] and dist[v] < best:
                    u, best = v, dist[v]
            in_tree[u] = True
            total += best
            ux, uy = points[u]
            for v in range(n):
                if not in_tree[v]:
                    d = abs(ux - points[v][0]) + abs(uy - points[v][1])
                    if d < dist[v]:
                        dist[v] = d
        return total

sol = Solution()
for pts, want in examples:
    print(sol.minCostConnectPoints(pts), "| mong đợi", want)

20 | mong đợi 20
18 | mong đợi 18


## 10. Vì sao thuật toán đúng

**Tính chất lát cắt.** Giả sử `e = (a, b)` là cạnh nhẹ nhất băng qua lát cắt `(S, V\S)` và `T` là một MST không chứa `e`. Thêm `e` vào `T` tạo đúng một chu trình; chu trình này phải băng qua lát cắt ít nhất hai lần, nên có cạnh `f ≠ e` cũng băng qua. Vì `w(e) ≤ w(f)`, cây `T + e − f` là cây khung có trọng số `≤ w(T)` ⇒ cũng là MST và chứa `e`.

**Prim.** Bất biến: tập cạnh đã chọn là tập con của một MST. Mỗi bước, `dist[u]` nhỏ nhất ứng với cạnh nhẹ nhất băng qua lát cắt `(cây hiện tại, phần còn lại)` ⇒ theo tính chất lát cắt, thêm nó vẫn giữ bất biến. Sau `n` bước có cây khung ⇒ là MST.

**Kruskal.** Khi xét cạnh `(i, j)` nối hai thành phần khác nhau `A ∋ i`, nó là cạnh nhẹ nhất băng qua lát cắt `(A, V\A)` (mọi cạnh nhẹ hơn đã xét và không băng qua — nếu băng qua thì đã được nhận hoặc nằm trong cùng thành phần). Cạnh nối cùng thành phần tạo chu trình nên bị loại đúng.

**Prim heap (lazy).** Heap luôn chứa (ít nhất) bản ghi đúng `(dist[v], v)` cho mỗi `v` chưa vào cây; bản ghi cũ có khoá lớn hơn hoặc trỏ tới đỉnh đã vào cây và bị bỏ qua ⇒ phần tử hợp lệ nhỏ nhất pop ra đúng như bản mảng.

## 11. Phân tích Time / Space

Gọi `n` = số điểm, `m = n(n−1)/2` = số cạnh của đồ thị đầy đủ.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn tập cạnh | `O(C(m, n−1) · n)` | `O(m)` |
| 2. Prim mảng | `O(n²)` | `O(n)` |
| 3. Kruskal + DSU | `O(m log m) = O(n² log n)` | `O(m) = O(n²)` |
| 4. Prim heap (lazy) | `O(m log m) = O(n² log n)` xấu nhất | `O(m)` xấu nhất |

Với `n = 1000`: Prim mảng ~`10⁶` bước cơ bản; Kruskal phải tạo và sắp ~`5·10⁵` cạnh. Ngoài ra còn có các kỹ thuật chuyên cho Manhattan MST giảm số cạnh ứng viên xuống `O(n)` (xem phần 15).

## 12. Bộ test

In [17]:
ALL = [brute_force, mst_prim_array, mst_kruskal, mst_prim_heap, Solution().minCostConnectPoints]

for f in ALL:
    for pts, want in examples:
        assert f([list(p) for p in pts]) == want, (f.__name__, pts)
print("Ví dụ đề: OK")

edge = [
    ([[0, 0]], 0),                                  # 1 điểm
    ([[0, 0], [1, 1]], 2),                          # 2 điểm
    ([[-1000000, -1000000], [1000000, 1000000]], 4000000),   # toạ độ biên
    ([[0, 0], [1, 1], [1, 0], [-1, 1]], 4),
    ([[0, 0], [0, 1], [0, 2], [0, 3]], 3),          # thẳng hàng
]
for f in ALL:
    for pts, want in edge:
        assert f([list(p) for p in pts]) == want, (f.__name__, pts, f([list(p) for p in pts]))
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [18]:
# Ngẫu nhiên seed cố định: đối chiếu với vét cạn (n ≤ 6), điểm phân biệt
def random_points(n, lo, hi, rng):
    s = set()
    while len(s) < n:
        s.add((rng.randint(lo, hi), rng.randint(lo, hi)))
    return [list(p) for p in s]

rng = random.Random(1584)
for _ in range(400):
    n = rng.randint(1, 6)
    pts = random_points(n, -5, 5, rng)
    want = brute_force(pts)
    for f in ALL[1:]:
        assert f(pts) == want, (f.__name__, pts)
print("400 bộ nhỏ khớp vét cạn: OK")

# Bộ vừa: các cách nhanh đối chiếu lẫn nhau
for _ in range(200):
    n = rng.randint(1, 60)
    pts = random_points(n, -10**6, 10**6, rng)
    res = {f(pts) for f in ALL[1:]}
    assert len(res) == 1
print("200 bộ vừa: các cách nhanh khớp nhau")

400 bộ nhỏ khớp vét cạn: OK
200 bộ vừa: các cách nhanh khớp nhau


In [19]:
# Kiểm tra tính chất: tổng MST không đổi khi tịnh tiến mọi điểm, và luôn >= (max dist từ 1 điểm tới gần nhất)
rng = random.Random(7)
for _ in range(100):
    pts = random_points(rng.randint(2, 30), -100, 100, rng)
    dx, dy = rng.randint(-50, 50), rng.randint(-50, 50)
    moved = [[x + dx, y + dy] for x, y in pts]
    a, b = mst_prim_array(pts), mst_prim_array(moved)
    assert a == b
    nearest = max(min(manhattan(p, q) for q in pts if q is not p) for p in pts)
    assert a >= nearest
print("Tính chất tịnh tiến & cận dưới: OK")

Tính chất tịnh tiến & cận dưới: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — "Lấy `n − 1` cạnh rẻ nhất" mà không kiểm tra chu trình

Các cạnh rẻ nhất có thể tạo chu trình trong một cụm và bỏ rơi điểm ở xa.

In [20]:
def wrong_no_cycle_check(points):
    n = len(points)
    edges = sorted(manhattan(points[i], points[j]) for i, j in combinations(range(n), 2))
    return sum(edges[:n - 1])                               # SAI: không kiểm chu trình/liên thông

pts = [[0, 0], [1, 0], [0, 1], [100, 100]]
print("sai:", wrong_no_cycle_check(pts), "| đúng:", Solution().minCostConnectPoints(pts))

sai: 4 | đúng: 201


### Lỗi 2 — Prim: chọn min mà **không loại** các đỉnh đã vào cây

Đỉnh đã vào cây có `dist = 0` (đỉnh gốc) hoặc giá trị nhỏ ⇒ bị chọn lại liên tục.

In [21]:
def wrong_prim_no_visited(points):
    n = len(points)
    dist = [float("inf")] * n
    dist[0] = 0
    in_tree = [False] * n
    total = 0
    for _ in range(n):
        u = min(range(n), key=lambda v: dist[v])          # SAI: thiếu điều kiện "not in_tree[v]"
        in_tree[u] = True
        total += dist[u]
        for v in range(n):
            if not in_tree[v]:
                dist[v] = min(dist[v], manhattan(points[u], points[v]))
    return total

pts = examples[0][0]
print("sai:", wrong_prim_no_visited(pts), "| đúng:", Solution().minCostConnectPoints(pts))

sai: 0 | đúng: 20


### Lỗi 3 — Dùng khoảng cách Euclid (hoặc quên `abs`)

In [22]:
def wrong_distance(points):
    n = len(points)
    dist = [float("inf")] * n
    dist[0] = 0
    in_tree = [False] * n
    total = 0
    for _ in range(n):
        u = min((v for v in range(n) if not in_tree[v]), key=lambda v: dist[v])
        in_tree[u] = True
        total += dist[u]
        for v in range(n):
            if not in_tree[v]:
                d = (points[u][0] - points[v][0]) + (points[u][1] - points[v][1])   # SAI: quên abs
                dist[v] = min(dist[v], d)
    return total

for pts, _ in examples:
    print("sai:", wrong_distance(pts), "| đúng:", Solution().minCostConnectPoints(pts))

sai: -31 | đúng: 20
sai: 18 | đúng: 18


### Lỗi 4 — Union-Find: gộp **đỉnh** thay vì gộp **gốc**

`parent[i] = j` (thay vì `parent[find(i)] = find(j)`) làm mất liên kết của các đỉnh khác trong cây cũ ⇒ không phát hiện chu trình đúng.

In [23]:
def wrong_union(points):
    n = len(points)
    edges = sorted((manhattan(points[i], points[j]), i, j) for i, j in combinations(range(n), 2))
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    total = used = 0
    for w, i, j in edges:
        if find(i) != find(j):
            parent[i] = j                                   # SAI: phải là parent[find(i)] = find(j)
            total += w
            used += 1
            if used == n - 1:
                break
    return total

rng = random.Random(11)
for _ in range(300):
    pts = random_points(6, -10, 10, rng)
    if wrong_union(pts) != Solution().minCostConnectPoints(pts):
        print("phản ví dụ:", pts)
        print("sai:", wrong_union(pts), "| đúng:", Solution().minCostConnectPoints(pts))
        break

phản ví dụ: [[4, 4], [6, 5], [6, 8], [10, 9], [-4, -5], [4, 7]]
sai: 17 | đúng: 31


### Lỗi 5 — Prim heap: không bỏ qua bản ghi cũ của đỉnh đã vào cây

Một đỉnh có thể nằm trong heap nhiều lần (với khoảng cách cũ lớn hơn). Nếu không kiểm tra `in_tree[u]` khi pop, nó được cộng thêm lần nữa.

In [24]:
def wrong_heap_no_skip(points):
    n = len(points)
    dist = [float("inf")] * n
    dist[0] = 0
    in_tree = [False] * n
    heap = [(0, 0)]
    total = 0
    while heap:
        d, u = heapq.heappop(heap)
        in_tree[u] = True                                   # SAI: thiếu "if in_tree[u]: continue"
        total += d
        for v in range(n):
            if not in_tree[v]:
                nd = manhattan(points[u], points[v])
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(heap, (nd, v))
    return total

pts = examples[0][0]
print("sai:", wrong_heap_no_skip(pts), "| đúng:", Solution().minCostConnectPoints(pts))

sai: 47 | đúng: 20


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Nối tất cả với chi phí nhỏ nhất", "đúng một đường đi giữa mọi cặp"** ⇒ **MST**.
- Đồ thị **đầy đủ / dày** (trọng số tính từ toạ độ) ⇒ **Prim mảng `O(n²)`**, không cần lưu cạnh.
- Đề **cho sẵn danh sách cạnh**, thưa ⇒ **Kruskal + DSU** (hoặc Prim heap).
- Luôn nhớ: cây khung có đúng `n − 1` cạnh; dừng Kruskal sớm khi đủ.
- DSU: luôn gộp **gốc**; thêm nén đường đi + hợp theo kích thước.
- Kiểm tra nhanh: 1 điểm → `0`; 2 điểm → khoảng cách của chúng; các điểm thẳng hàng cách đều 1 → `n − 1`.

## 15. Biến thể của bài

1. **LC 1135 — Connecting Cities With Minimum Cost:** cho sẵn cạnh, trả `-1` nếu không liên thông (bài tập 5).
2. **LC 1168 — Optimize Water Distribution in a Village:** thêm **đỉnh ảo** nối tới mọi nhà với chi phí đào giếng.
3. **LC 1489 — Find Critical and Pseudo-Critical Edges in MST:** chạy Kruskal nhiều lần, bắt buộc/loại bỏ từng cạnh.
4. **Cây khung lớn nhất** (maximum spanning tree): sắp giảm dần.
5. **Manhattan MST `O(n log n)`:** chỉ cần xét, với mỗi điểm, điểm gần nhất trong mỗi góc 45° (8 góc, đối xứng còn 4) ⇒ `O(n)` cạnh ứng viên, rồi Kruskal. Kỹ thuật nâng cao, dùng BIT/sweep line.
6. **Borůvka:** mỗi vòng, mỗi thành phần chọn cạnh rẻ nhất ra ngoài; `O(log n)` vòng — dễ song song hoá.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi tập `n−1` cạnh | hàm mũ | `O(m)` | kiểm thử, `n ≤ 6–7` |
| 2. Prim mảng | mở rộng cây bằng cạnh rẻ nhất, quét `dist` | `O(n²)` | `O(n)` | **đồ thị dày** — lời giải tốt nhất cho bài này |
| 3. Kruskal + DSU | sắp cạnh, thêm nếu không tạo chu trình | `O(n² log n)` | `O(n²)` | đề cho danh sách cạnh thưa; dễ nhớ |
| 4. Prim heap | Prim với hàng đợi ưu tiên, bỏ bản ghi cũ | `O(n² log n)` | `O(n²)` | đồ thị thưa có danh sách kề |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minCostConnectPoints([[0,0],[1,1],[1,0],[-1,1]])` trả về bao nhiêu?

**Bài 2 (tính tay):** Với 4 điểm `A(0,0)`, `B(2,0)`, `C(2,3)`, `D(0,3)`, liệt kê 6 cạnh theo thứ tự Kruskal xét và cho biết cạnh nào được nhận. Tổng MST?

**Bài 3 (điền TODO):** Viết `ex3_mst_edges(points)` trả về **danh sách cạnh** `(i, j, w)` của một MST (dùng Prim mảng có mảng `parent`).

**Bài 4 (tự viết lại):** Tự viết lại Kruskal **không dùng lớp DSU** (chỉ mảng `parent`, `size` và hàm `find` đệ quy có nén đường đi).

**Bài 5 (mini challenge — LC 1135):** Viết `ex5_min_cost_cities(n, connections)` với các thành phố đánh số `1..n` và `connections[i] = [a, b, cost]`; trả về chi phí nhỏ nhất để nối mọi thành phố, hoặc `-1` nếu không thể.

In [25]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_mst_edges(points):
    n = len(points)
    dist = [float("inf")] * n
    parent = [-1] * n
    dist[0] = 0
    in_tree = [False] * n
    edges = []
    # TODO: Prim mảng; khi cập nhật dist[v] thì cũng cập nhật parent[v] = u
    # TODO: khi đưa u (khác gốc) vào cây, thêm (parent[u], u, dist[u]) vào edges
    return edges

In [26]:
# Bài 5 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex5_min_cost_cities(n, connections):
    # TODO: sắp connections theo cost
    # TODO: Kruskal với DSU trên đỉnh 1..n
    # TODO: nếu số cạnh nhận được < n - 1 -> trả -1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tính các khoảng cách nhỏ: `(0,0)-(1,0)=1`, `(1,0)-(1,1)=1`. (2) `(-1,1)` gần nhất với `(0,0)` hoặc `(1,1)`: khoảng cách 2. (3) Tổng `1 + 1 + 2`.

**Bài 2.** (1) Cạnh: AB=2, CD=2, AD=3, BC=3, AC=5, BD=5. (2) Nhận AB, CD, rồi AD (nối hai cụm), BC tạo chu trình. (3) Tổng `2 + 2 + 3 = 7`.

**Bài 3.** (1) Thêm `parent`. (2) Cạnh của đỉnh `u` là `(parent[u], u)`. (3) Kiểm tra: `n − 1` cạnh, tổng bằng đáp án, liên thông.

**Bài 4.** (1) `find(x)`: nếu `parent[x] != x` thì `parent[x] = find(parent[x])`. (2) Hợp theo `size`. (3) Đệ quy sâu tối đa `O(log n)` nhờ hợp theo kích thước — an toàn.

**Bài 5.** (1) Đỉnh từ 1 nên tạo DSU kích thước `n + 1`. (2) Đếm số cạnh được nhận. (3) Có thể có nhiều cạnh giữa cùng cặp đỉnh — Kruskal tự xử lý.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
ans1 = Solution().minCostConnectPoints([[0, 0], [1, 1], [1, 0], [-1, 1]])
print("Bài 1:", ans1)
assert ans1 == 4

# Bài 2
names = "ABCD"
P = [[0, 0], [2, 0], [2, 3], [0, 3]]
print("Bài 2 — thứ tự Kruskal:")
total2 = trace_kruskal(P)
assert total2 == 7
print("Bài 1, 2: OK")

Bài 1: 4
Bài 2 — thứ tự Kruskal:
cạnh P0-P1 w= 2: NHẬN | total=2 | số cạnh=1
cạnh P2-P3 w= 2: NHẬN | total=4 | số cạnh=2
cạnh P0-P3 w= 3: NHẬN | total=7 | số cạnh=3
Bài 1, 2: OK


In [28]:
# Bài 3
def ex3_mst_edges_sol(points):
    n = len(points)
    dist, parent = [float("inf")] * n, [-1] * n
    dist[0] = 0
    in_tree = [False] * n
    edges = []
    for _ in range(n):
        u = min((v for v in range(n) if not in_tree[v]), key=lambda v: dist[v])
        in_tree[u] = True
        if parent[u] != -1:
            edges.append((parent[u], u, dist[u]))
        for v in range(n):
            if not in_tree[v]:
                d = manhattan(points[u], points[v])
                if d < dist[v]:
                    dist[v], parent[v] = d, u
    return edges

print("Bài 3:", ex3_mst_edges_sol(examples[0][0]))
rng = random.Random(3)
for _ in range(200):
    pts = random_points(rng.randint(1, 25), -50, 50, rng)
    E = ex3_mst_edges_sol(pts)
    assert len(E) == len(pts) - 1
    assert sum(w for _, _, w in E) == Solution().minCostConnectPoints(pts)
    d = DSU(len(pts))
    assert all(d.union(i, j) for i, j, _ in E)            # không chu trình => là cây
    assert all(manhattan(pts[i], pts[j]) == w for i, j, w in E)
print("Bài 3: OK")

Bài 3: [(0, 1, 4), (1, 3, 3), (3, 4, 4), (1, 2, 9)]


Bài 3: OK


In [29]:
# Bài 4
def ex4_kruskal_plain(points):
    n = len(points)
    parent, size = list(range(n)), [1] * n

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    edges = sorted((manhattan(points[i], points[j]), i, j) for i in range(n) for j in range(i + 1, n))
    total = used = 0
    for w, i, j in edges:
        ri, rj = find(i), find(j)
        if ri == rj:
            continue
        if size[ri] < size[rj]:
            ri, rj = rj, ri
        parent[rj] = ri
        size[ri] += size[rj]
        total += w
        used += 1
        if used == n - 1:
            break
    return total

rng = random.Random(4)
for _ in range(200):
    pts = random_points(rng.randint(1, 40), -1000, 1000, rng)
    assert ex4_kruskal_plain(pts) == mst_prim_array(pts)
print("Bài 4: OK, ví dụ:", [ex4_kruskal_plain(p) for p, _ in examples])

Bài 4: OK, ví dụ: [20, 18]


In [30]:
# Bài 5 — LC 1135
def ex5_min_cost_cities_sol(n, connections):
    d = DSU(n + 1)
    total = used = 0
    for a, b, c in sorted(connections, key=lambda e: e[2]):
        if d.union(a, b):
            total += c
            used += 1
            if used == n - 1:
                break
    return total if used == n - 1 else -1

def ex5_brute(n, connections):
    if n == 1:
        return 0
    best = -1
    for subset in combinations(connections, n - 1):
        d = DSU(n + 1)
        if all(d.union(a, b) for a, b, _ in subset):
            cost = sum(c for _, _, c in subset)
            best = cost if best == -1 else min(best, cost)
    return best

print("Bài 5:", ex5_min_cost_cities_sol(3, [[1, 2, 5], [1, 3, 6], [2, 3, 1]]),
      ex5_min_cost_cities_sol(4, [[1, 2, 3], [3, 4, 4]]))
assert ex5_min_cost_cities_sol(3, [[1, 2, 5], [1, 3, 6], [2, 3, 1]]) == 6
assert ex5_min_cost_cities_sol(4, [[1, 2, 3], [3, 4, 4]]) == -1
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 5)
    conns = [[rng.randint(1, n), rng.randint(1, n), rng.randint(1, 20)] for _ in range(rng.randint(0, 7))]
    conns = [c for c in conns if c[0] != c[1]]
    assert ex5_min_cost_cities_sol(n, conns) == ex5_brute(n, conns), (n, conns)
print("Bài 5: OK")

Bài 5: 6 -1
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Prim O(n^2) cho đồ thị dày (trọng số tính từ toạ độ)
def prim(points):
    n = len(points)
    dist = [float("inf")] * n
    dist[0] = 0
    in_tree = [False] * n
    total = 0
    for _ in range(n):
        u = min((v for v in range(n) if not in_tree[v]), key=dist.__getitem__)
        in_tree[u] = True
        total += dist[u]
        for v in range(n):
            if not in_tree[v]:
                dist[v] = min(dist[v], abs(points[u][0] - points[v][0]) + abs(points[u][1] - points[v][1]))
    return total

# Kruskal: sort cạnh -> DSU.union(i, j) thành công thì nhận -> đủ n-1 cạnh thì dừng
```

**Checklist trước khi submit:**
- [ ] Khoảng cách Manhattan có `abs` cho cả hai trục.
- [ ] Prim: chỉ chọn đỉnh **chưa vào cây**; chỉ cập nhật đỉnh chưa vào cây.
- [ ] Kruskal: kiểm chu trình bằng DSU, gộp **gốc**, dừng khi đủ `n − 1` cạnh.
- [ ] Prim heap: `if in_tree[u]: continue` sau khi pop.
- [ ] `n = 1` trả `0`.
- [ ] Thử nhanh: ví dụ → `20, 18`; `[[0,0],[1,1]]` → `2`.